In [11]:
import torch.nn as nn
import torch

class BasicBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_ch)

        self.downsample = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)

class CEEDNet1DResNet18(nn.Module):
    def __init__(self, in_channels=10, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 64, blocks=2)
        self.layer2 = self._make_layer(64, 128, blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=2, stride=2)

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def _make_layer(self, in_ch, out_ch, blocks, stride=1):
        layers = [BasicBlock1D(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_avg_pool(x).squeeze(-1)
        return self.fc(x)


In [12]:
def preprocess(csv_path):
    df = pd.read_csv(csv_path)  # assumes channel names as headers
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)
    data = df.values.T

    if data.shape[1] > 2000:
        data = data[:, :2000]
    else:
        pad = 2000 - data.shape[1]
        data = np.pad(data, ((0, 0), (0, pad)), mode='constant')

    mean = np.mean(data, axis=1, keepdims=True)
    std = np.std(data, axis=1, keepdims=True)
    z = (data - mean) / (std + 1e-8)

    return torch.tensor(z, dtype=torch.float32).unsqueeze(0)  # shape: [1, 10, 2000]


In [13]:
model = CEEDNet1DResNet18()
model.load_state_dict(torch.load("/kaggle/input/pretrained50eeg/pytorch/default/1/preEEGmodel.pt", map_location="cpu"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
#model.eval()
model.train()

In [ ]:
import os
from tqdm import tqdm

test_folder = "/kaggle/input/reduced-signal-csv/signal_csv_reduced"
results = []

for filename in tqdm(os.listdir(test_folder)):
    if filename.endswith(".csv"):
        file_path = os.path.join(test_folder, filename)

        try:
            x = preprocess(file_path).to(device)
            with torch.no_grad():
                output = model(x)
                pred = torch.argmax(output, dim=1).item()

            true_label = int(filename.split("_")[0])
            results.append((filename, true_label, pred))

        except Exception as e:
            print(f"Error in {filename}: {e}")


 64%|██████▎   | 704/1106 [03:19<01:55,  3.47it/s]

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Unpack results
filenames, y_true, y_pred = zip(*results)

print(f"✅ Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\n📊 Classification Report:\n", classification_report(y_true, y_pred))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1,2], yticklabels=[0,1,2])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [16]:
df = pd.DataFrame(results, columns=["Filename", "True_Label", "Predicted_Label"])
df.to_csv("test_predictions.csv", index=False)
print("Saved results to test_predictions.csv")


Saved results to test_predictions.csv
